# Managing the SCF GLM Model Lifecycle in Unity Catalog

This notebook walks through the complete MLOps model lifecycle for the SCF Cohort GLM model,
following the Databricks recommended pattern:

```
Data Prep & Featurization
        │
        ▼
  Train Model (GLM per product, MLflow autologging)
        │
        ▼
  Promote New Model Version
  Search best run → Register @challenger → Tag + Describe
        │
        ▼
  Model Validation Job
  Evaluate challenger MAPE vs champion
        │
   ┌────┴────┐
   │         │
Approved  Rejected
   │         │
@champion  archived
   │
   ▼
Batch Inference → Inference Tables (UC)
```

**Aliases used:**
- `@baseline`   — first-ever registered version (reference point)
- `@challenger` — newest candidate model under evaluation
- `@champion`   — currently deployed model serving production inference

---
Last environment tested: `mlflow>=2.16`, Databricks Runtime 14.3 LTS+

## 0 — Setup
Configure catalog, schema, and model name. These match the DAB variables in `databricks.yml`.

In [ ]:
import sys
sys.path.append("..")

import mlflow
from mlflow import MlflowClient

# ── Configure these for your environment ──────────────────────────────────
CATALOG    = "pd_dtl_ds_dev"          # dev catalog
SCHEMA     = "savings_cashflow"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.scf_cohort_model"
EXPERIMENT = f"/Shared/scf_cohort/{CATALOG}_training"
# ──────────────────────────────────────────────────────────────────────────

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()
print(f"Model: {MODEL_NAME}")
print(f"Experiment: {EXPERIMENT}")

## 1 — Discover: List All Registered Versions

Unity Catalog Model Registry stores every registered version with its aliases and tags.
Users with `USE CATALOG` + `EXECUTE` privilege on the model can discover and inspect it.

In [ ]:
import pandas as pd

try:
    rm = client.get_registered_model(MODEL_NAME)
    print(f"Model: {rm.name}")
    print(f"Description: {rm.description}")
    print(f"Aliases: {rm.aliases}")
    print()

    versions = client.search_model_versions(f"name='{MODEL_NAME}'")
    rows = []
    for v in versions:
        alias_list = [a for a, ver in rm.aliases.items() if ver == v.version]
        rows.append({
            "version":     v.version,
            "aliases":     ", ".join(alias_list) or "—",
            "stage":       v.tags.get("stage_history", "—"),
            "trained_at":  v.tags.get("trained_at", "—"),
            "cutoff":      v.tags.get("cutoff_period", "—"),
            "mape":        v.tags.get("portfolio_mape", "—"),
            "n_products":  v.tags.get("n_products", "—"),
            "description": (v.description or "")[:80],
        })

    df = pd.DataFrame(rows).sort_values("version", ascending=False)
    display(df)
except Exception as e:
    print(f"Model not yet registered: {e}")
    print("Run the training pipeline first: databricks bundle run scf_training_pipeline -t dev")

## 2 — Promote New Model Version

Programmatically find the best training run from the MLflow experiment
and register it as a new `@challenger`. This mirrors the **ML Engineer** step
in the lifecycle diagram: *Search runs → log best model to UC*.

In [ ]:
from src.common import mlflow_utils

# Search for the best run by portfolio MAPE (lowest wins)
try:
    best_run_id = mlflow_utils.search_best_run(
        experiment_name=EXPERIMENT,
        metric_key="portfolio_mape_balance",
    )
    print(f"Best run ID: {best_run_id}")
    run = client.get_run(best_run_id)
    print(f"Metrics: {run.data.metrics}")
    print(f"Params:  {run.data.params}")
except Exception as e:
    print(f"Could not find best run: {e}")

In [ ]:
# Register the best run as a new version → alias @challenger
# (In the automated pipeline this is done by 05_model_registration.py)

# To manually register from a specific run:
# version = mlflow_utils.register_challenger(best_run_id, "model", MODEL_NAME)
# print(f"Registered version {version} as @challenger")

# Show current @challenger
try:
    challenger = client.get_model_version_by_alias(MODEL_NAME, "challenger")
    print(f"Current @challenger: v{challenger.version}")
    print(f"  Description : {challenger.description}")
    print(f"  Tags        : {challenger.tags}")
except Exception:
    print("No @challenger alias found yet.")

## 3 — Tag and Document the Challenger Version

After registration, add rich metadata tags and a description.
These are visible in **Unity Catalog Explorer → Catalog > model_name > version**.

In [ ]:
# View tags currently on the challenger version
try:
    challenger = client.get_model_version_by_alias(MODEL_NAME, "challenger")
    version = challenger.version

    print(f"Version {version} tags:")
    for k, v in challenger.tags.items():
        print(f"  {k:30s}: {v}")

    print(f"\nVersion {version} description:")
    print(f"  {challenger.description}")
except Exception as e:
    print(e)

In [ ]:
# Manually add/update a tag on the challenger (e.g. for annotation)
# client.set_model_version_tag(MODEL_NAME, version, "reviewed_by", "your-name")
# client.set_model_version_tag(MODEL_NAME, version, "review_date", "2026-07-14")

# Update description of the registered model
# mlflow_utils.set_registered_model_description(
#     MODEL_NAME,
#     "Updated description - reviewed and approved by risk team."
# )
print("Uncomment lines above to manually tag or describe a version.")

## 4 — Model Validation: Champion vs Challenger

The **Model Validation Job** (`04_model_evaluation.py` + `05_model_registration.py`)
automatically compares challenger MAPE vs champion MAPE. Here we inspect the outcome manually.

In [ ]:
import pandas as pd

def show_alias(alias: str):
    try:
        mv = client.get_model_version_by_alias(MODEL_NAME, alias)
        run = client.get_run(mv.run_id)
        mape = run.data.metrics.get("portfolio_mape_balance", "N/A")
        print(f"@{alias:12s}  v{mv.version:4s}  MAPE={mape}  stage={mv.tags.get('stage_history','?')}")
        return float(mape) if mape != "N/A" else None
    except Exception:
        print(f"@{alias:12s}  (not found)")
        return None

print("── Current Model Lifecycle State ──────────────────")
baseline_mape   = show_alias("baseline")
challenger_mape = show_alias("challenger")
champion_mape   = show_alias("champion")

if champion_mape and challenger_mape:
    pct_improvement = (champion_mape - challenger_mape) / champion_mape * 100
    print(f"\nChallenger improvement over champion: {pct_improvement:.1f}%")
    threshold = 15.0
    verdict = "APPROVED → promote" if pct_improvement >= threshold else f"REJECTED (threshold {threshold}%)"
    print(f"Verdict (threshold={threshold}%): {verdict}")

## 5 — Manual Promotion or Archive (Override)

In exceptional cases (e.g. regulatory sign-off), a human can manually promote
the challenger to champion, or archive a version, using the cells below.

> **Warning:** Manual promotion bypasses the automated MAPE gate. Document the reason.

In [ ]:
# ── MANUAL PROMOTION (uncomment and run if approved by risk team) ─────────

# promoted_v = mlflow_utils.promote_challenger_to_champion(MODEL_NAME)
# client.set_model_version_tag(MODEL_NAME, str(promoted_v), "stage_history", "champion")
# client.set_model_version_tag(MODEL_NAME, str(promoted_v), "promoted_by", "manual-override")
# client.set_model_version_tag(MODEL_NAME, str(promoted_v), "promotion_reason",
#                               "Approved by risk team - regulatory deadline")
# print(f"Manually promoted v{promoted_v} to @champion")

# ── MANUAL ARCHIVE (uncomment to archive a specific version) ─────────────

# VERSION_TO_ARCHIVE = "3"
# client.set_model_version_tag(MODEL_NAME, VERSION_TO_ARCHIVE, "stage_history", "archived")
# client.set_model_version_tag(MODEL_NAME, VERSION_TO_ARCHIVE, "archived_reason", "manual-review")
# print(f"Archived version {VERSION_TO_ARCHIVE}")

print("Uncomment the block you need and run this cell.")

## 6 — Secure: Permissions and Governance

Unity Catalog RBAC controls who can register, promote, or execute the model.
Run the cells below to inspect or grant permissions.

In [ ]:
# Show who has permissions on the registered model
# (requires workspace admin or model owner)
try:
    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()
    # UC registered model permissions
    perms = w.grants.get_effective(
        securable_type="REGISTERED_MODEL",
        full_name=MODEL_NAME,
    )
    for p in perms.privilege_assignments:
        print(f"  {p.principal:40s} → {[x.privilege for x in p.privileges]}")
except Exception as e:
    print(f"Cannot list permissions: {e}")

In [ ]:
# Grant EXECUTE on the model to a group (e.g. ML engineers for inference)
# Run as workspace admin or catalog owner:
#
# spark.sql(f"""
#     GRANT EXECUTE ON REGISTERED MODEL {MODEL_NAME}
#     TO `ml-engineers`
# """)
#
# Grant SELECT on inference output table to analysts:
# spark.sql(f"""
#     GRANT SELECT ON TABLE {CATALOG}.{SCHEMA}.inference_predictions
#     TO `data-analysts`
# """)
print("Uncomment the GRANT statements above and run as admin.")

## 7 — Full Lifecycle Summary

Shows all versions with their lifecycle stage in one table.

In [ ]:
try:
    rm = client.get_registered_model(MODEL_NAME)
    versions = client.search_model_versions(f"name='{MODEL_NAME}'")

    rows = []
    for v in sorted(versions, key=lambda x: int(x.version), reverse=True):
        alias_list = [a for a, ver in (rm.aliases or {}).items() if ver == v.version]
        stage = v.tags.get("stage_history", "registered")
        icon = {
            "champion":   "🏆",
            "challenger": "⚔️",
            "baseline":   "📌",
            "archived":   "🗄️",
            "rejected":   "❌",
        }.get(stage, "📦")
        rows.append({
            "v":           v.version,
            "stage":       f"{icon} {stage}",
            "aliases":     ", ".join(f"@{a}" for a in alias_list) or "—",
            "cutoff":      v.tags.get("cutoff_period", "—"),
            "mape":        v.tags.get("portfolio_mape", "—"),
            "n_products":  v.tags.get("n_products", "—"),
            "trained_at":  (v.tags.get("trained_at") or "")[:19],
        })

    display(pd.DataFrame(rows))
except Exception as e:
    print(f"Run training pipeline first: {e}")